# Análise Exploratória de Dados (EDA) - Dataset SEER Clínico
## Tech Challenge Fase 1 - Saúde e Segurança da Mulher
**FIAP POS TECH - IADT**

**Responsável pela EDA:** Natalia Cabrera (Desenvolvedora)

### Objetivo
Análise exploratória completa do Dataset Principal Clínico (SEER Breast Cancer),
contendo 4.024 registros de pacientes com câncer de mama. O objetivo é entender
padrões clínicos, distribuições e correlações que auxiliem na modelagem preditiva
de sobrevivência.

### Dataset
- **Fonte:** SEER (Surveillance, Epidemiology, and End Results Program)
- **Registros:** 4.024 pacientes
- **Target:** Status (Alive/Dead) - sobrevivência da paciente
- **Features:** Idade, raça, estágio do tumor, grau, tamanho, receptores hormonais, linfonodos

In [ ]:
# Imports e configurações
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configurações de visualização
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')
sns.set_palette('Set2')

print('Bibliotecas carregadas com sucesso!')

## 1. Carregamento dos Dados

In [ ]:
# Carregar dataset SEER Clínico
df = pd.read_csv('../../data/raw/DatasetPrincipalClinico.csv', 
                 sep=';', encoding='latin-1')

# Remover coluna completamente nula
df = df.drop(columns=['Column1'])

print(f'Dataset carregado: {df.shape[0]} registros x {df.shape[1]} features')
print(f'Período de sobrevivência: {df["Survival Months"].min()} a {df["Survival Months"].max()} meses')
df.head()

## 2. Visão Geral do Dataset

In [ ]:
# Informações gerais
print('=' * 60)
print('INFORMAÇÕES DO DATASET SEER CLÍNICO')
print('=' * 60)
print(f'\nTotal de registros: {len(df)}')
print(f'Total de features: {df.shape[1]}')
print(f'\nTipos de dados:')
print(df.dtypes.value_counts())
print(f'\nValores nulos por coluna:')
print(df.isnull().sum())
print(f'\nDuplicatas: {df.duplicated().sum()}')

In [ ]:
# Estatísticas descritivas - variáveis numéricas
numeric_cols = ['Age', 'Tumor Size', 'Regional Node Examined', 
                'Reginal Node Positive', 'Survival Months']
df[numeric_cols].describe().round(2)

In [ ]:
# Estatísticas descritivas - variáveis categóricas
cat_cols = ['Race', 'Marital Status', 'T Stage', 'N Stage', 
            '6th Stage', 'Grade', 'A Stage', 'Estrogen Status', 
            'Progesterone Status', 'Status']

for col in cat_cols:
    print(f'\n--- {col} ---')
    vc = df[col].value_counts()
    pct = df[col].value_counts(normalize=True) * 100
    summary = pd.DataFrame({'Contagem': vc, '%': pct.round(1)})
    print(summary)

## 3. Análise da Variável Target (Status)

In [ ]:
# Distribuição do target
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico de barras
colors = ['#66c2a5', '#fc8d62']
status_counts = df['Status'].value_counts()
axes[0].bar(status_counts.index, status_counts.values, color=colors, edgecolor='black')
axes[0].set_title('Distribuição do Status (Sobrevivência)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Número de Pacientes')
for i, (label, val) in enumerate(status_counts.items()):
    axes[0].text(i, val + 30, f'{val}\n({val/len(df)*100:.1f}%)', 
                ha='center', fontsize=12, fontweight='bold')

# Gráfico de pizza
axes[1].pie(status_counts.values, labels=['Viva', 'Óbito'], 
           autopct='%1.1f%%', colors=colors, startangle=90,
           textprops={'fontsize': 13}, explode=[0, 0.05])
axes[1].set_title('Proporção de Sobrevivência', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../../reports/figures/seer_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nDataset DESBALANCEADO: {status_counts["Alive"]/len(df)*100:.1f}% Vivas vs {status_counts["Dead"]/len(df)*100:.1f}% Óbito')
print(f'Proporção: {status_counts["Alive"]/status_counts["Dead"]:.1f}:1')

## 4. Distribuição das Variáveis Numéricas

In [ ]:
# Histogramas das variáveis numéricas com separação por Status
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    for status, color in zip(['Alive', 'Dead'], ['#66c2a5', '#fc8d62']):
        data = df[df['Status'] == status][col]
        label = 'Viva' if status == 'Alive' else 'Óbito'
        axes[i].hist(data, bins=30, alpha=0.6, color=color, label=label, edgecolor='black')
    axes[i].set_title(f'Distribuição de {col}', fontsize=13, fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frequência')
    axes[i].legend()

# Remover subplot extra
axes[5].set_visible(False)

plt.suptitle('Distribuição das Variáveis Numéricas por Status', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../../reports/figures/seer_numeric_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Boxplots comparativos por Status
fig, axes = plt.subplots(1, 5, figsize=(22, 6))

for i, col in enumerate(numeric_cols):
    sns.boxplot(data=df, x='Status', y=col, ax=axes[i], 
                palette=['#66c2a5', '#fc8d62'])
    axes[i].set_title(f'{col}', fontsize=12, fontweight='bold')
    axes[i].set_xticklabels(['Viva', 'Óbito'])

plt.suptitle('Comparação por Status de Sobrevivência', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../../reports/figures/seer_boxplots_by_status.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Análise das Variáveis Categóricas

In [ ]:
# Distribuição das categóricas com taxas de sobrevivência
cat_features = ['T Stage', 'N Stage', '6th Stage', 'Grade', 
                'Estrogen Status', 'Progesterone Status',
                'Race', 'A Stage']

fig, axes = plt.subplots(2, 4, figsize=(24, 12))
axes = axes.flatten()

for i, col in enumerate(cat_features):
    # Calcular taxa de óbito por categoria
    ct = pd.crosstab(df[col], df['Status'], normalize='index') * 100
    ct = ct.sort_values('Dead', ascending=True)
    
    ct.plot(kind='barh', stacked=True, ax=axes[i], 
            color=['#66c2a5', '#fc8d62'], edgecolor='black')
    axes[i].set_title(f'{col}', fontsize=13, fontweight='bold')
    axes[i].set_xlabel('% Pacientes')
    axes[i].legend(['Viva', 'Óbito'], loc='lower right')
    
    # Adicionar % de óbito
    for j, (idx, row) in enumerate(ct.iterrows()):
        axes[i].text(row['Dead'] + row['Alive'] + 1, j, 
                    f'{row["Dead"]:.1f}%', va='center', fontsize=9)

plt.suptitle('Taxa de Mortalidade por Variável Categórica', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../../reports/figures/seer_categorical_survival.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Análise de Correlação

In [ ]:
# Preparar dados numéricos para correlação
from sklearn.preprocessing import LabelEncoder

df_encoded = df.copy()
# Criar target binário
df_encoded['target'] = (df_encoded['Status'] == 'Alive').astype(int)

# Codificar categóricas
le = LabelEncoder()
for col in cat_cols:
    if col in df_encoded.columns and col != 'Status':
        df_encoded[col + '_encoded'] = le.fit_transform(df_encoded[col])

# Selecionar colunas numéricas para correlação
corr_cols = numeric_cols + [c + '_encoded' for c in cat_cols if c != 'Status'] + ['target']
corr_cols = [c for c in corr_cols if c in df_encoded.columns]

corr_matrix = df_encoded[corr_cols].corr()

# Heatmap
fig, ax = plt.subplots(figsize=(16, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', 
            cmap='RdBu_r', center=0, ax=ax,
            square=True, linewidths=0.5,
            vmin=-1, vmax=1)
ax.set_title('Matriz de Correlação - Dataset SEER Clínico', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../../reports/figures/seer_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Correlações com o target
print('\nCorrelação com o Target (Sobrevivência):')
print('=' * 50)
target_corr = corr_matrix['target'].drop('target').sort_values(key=abs, ascending=False)
for feat, val in target_corr.items():
    direction = '↑' if val > 0 else '↓'
    print(f'  {direction} {feat}: {val:.4f}')

## 7. Análise dos Receptores Hormonais (Saúde da Mulher)

In [ ]:
# Análise detalhada dos receptores hormonais - Estrogênio e Progesterona
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Estrogênio x Sobrevivência
ct1 = pd.crosstab(df['Estrogen Status'], df['Status'])
ct1.plot(kind='bar', ax=axes[0], color=['#fc8d62', '#66c2a5'], edgecolor='black')
axes[0].set_title('Estrogênio x Sobrevivência', fontsize=13, fontweight='bold')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)
axes[0].set_ylabel('Pacientes')
axes[0].legend(['Óbito', 'Viva'])

# Progesterona x Sobrevivência
ct2 = pd.crosstab(df['Progesterone Status'], df['Status'])
ct2.plot(kind='bar', ax=axes[1], color=['#fc8d62', '#66c2a5'], edgecolor='black')
axes[1].set_title('Progesterona x Sobrevivência', fontsize=13, fontweight='bold')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
axes[1].set_ylabel('Pacientes')
axes[1].legend(['Óbito', 'Viva'])

# Combinação ER/PR
df['ER_PR'] = df['Estrogen Status'] + ' / ' + df['Progesterone Status']
ct3 = pd.crosstab(df['ER_PR'], df['Status'], normalize='index') * 100
ct3.plot(kind='barh', stacked=True, ax=axes[2], 
         color=['#fc8d62', '#66c2a5'], edgecolor='black')
axes[2].set_title('Combinação ER/PR x Sobrevivência', fontsize=13, fontweight='bold')
axes[2].set_xlabel('% Pacientes')
axes[2].legend(['Óbito', 'Viva'])

plt.suptitle('Impacto dos Receptores Hormonais na Sobrevivência', 
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../../reports/figures/seer_hormonal_receptors.png', dpi=150, bbox_inches='tight')
plt.show()

# Taxas de sobrevivência por combinação
print('\nTaxa de Sobrevivência por Combinação Hormonal:')
print('=' * 50)
for combo in df['ER_PR'].unique():
    subset = df[df['ER_PR'] == combo]
    alive_pct = (subset['Status'] == 'Alive').mean() * 100
    print(f'  {combo}: {alive_pct:.1f}% sobrevivência (n={len(subset)})')

## 8. Análise de Estadiamento e Prognóstico

In [ ]:
# Relação entre estágios do tumor e sobrevivência
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# T Stage x Tumor Size
sns.boxplot(data=df, x='T Stage', y='Tumor Size', hue='Status', 
            ax=axes[0, 0], palette=['#66c2a5', '#fc8d62'],
            order=['T1', 'T2', 'T3', 'T4'])
axes[0, 0].set_title('Tamanho do Tumor por T Stage e Status', fontsize=13, fontweight='bold')
axes[0, 0].legend(title='Status', labels=['Viva', 'Óbito'])

# N Stage x Linfonodos positivos
sns.boxplot(data=df, x='N Stage', y='Reginal Node Positive', hue='Status',
            ax=axes[0, 1], palette=['#66c2a5', '#fc8d62'],
            order=['N1', 'N2', 'N3'])
axes[0, 1].set_title('Linfonodos Positivos por N Stage e Status', fontsize=13, fontweight='bold')
axes[0, 1].legend(title='Status', labels=['Viva', 'Óbito'])

# 6th Stage x Survival Months
stage_order = ['IIA', 'IIB', 'IIIA', 'IIIB', 'IIIC']
sns.boxplot(data=df, x='6th Stage', y='Survival Months', hue='Status',
            ax=axes[1, 0], palette=['#66c2a5', '#fc8d62'],
            order=stage_order)
axes[1, 0].set_title('Meses de Sobrevivência por Estágio e Status', fontsize=13, fontweight='bold')
axes[1, 0].legend(title='Status', labels=['Viva', 'Óbito'])

# Grade x Taxa de mortalidade
grade_surv = df.groupby('Grade')['Status'].apply(
    lambda x: (x == 'Dead').mean() * 100
).sort_values()
colors_grade = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(grade_surv)))
axes[1, 1].barh(range(len(grade_surv)), grade_surv.values, color=colors_grade, edgecolor='black')
axes[1, 1].set_yticks(range(len(grade_surv)))
axes[1, 1].set_yticklabels([g[:30] for g in grade_surv.index])
axes[1, 1].set_xlabel('Taxa de Mortalidade (%)')
axes[1, 1].set_title('Taxa de Mortalidade por Grau do Tumor', fontsize=13, fontweight='bold')
for i, v in enumerate(grade_surv.values):
    axes[1, 1].text(v + 0.5, i, f'{v:.1f}%', va='center', fontsize=11)

plt.suptitle('Análise de Estadiamento e Prognóstico', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../../reports/figures/seer_staging_prognosis.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Análise Demográfica

In [ ]:
# Análise por faixa etária e raça
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Idade x Status
for status, color, label in zip(['Alive', 'Dead'], ['#66c2a5', '#fc8d62'], ['Viva', 'Óbito']):
    data = df[df['Status'] == status]['Age']
    axes[0].hist(data, bins=20, alpha=0.6, color=color, label=label, edgecolor='black', density=True)
axes[0].set_title('Distribuição de Idade por Status', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Idade')
axes[0].set_ylabel('Densidade')
axes[0].legend()

# Raça x Taxa de sobrevivência
race_surv = df.groupby('Race')['Status'].apply(
    lambda x: (x == 'Alive').mean() * 100
).sort_values(ascending=True)
bars = axes[1].barh(range(len(race_surv)), race_surv.values, 
                    color=sns.color_palette('Set2', len(race_surv)), edgecolor='black')
axes[1].set_yticks(range(len(race_surv)))
axes[1].set_yticklabels([r[:25] for r in race_surv.index])
axes[1].set_xlabel('Taxa de Sobrevivência (%)')
axes[1].set_title('Sobrevivência por Raça', fontsize=13, fontweight='bold')
for i, v in enumerate(race_surv.values):
    axes[1].text(v + 0.3, i, f'{v:.1f}%', va='center', fontsize=11)

# Estado civil x Taxa de sobrevivência
marital_surv = df.groupby('Marital Status')['Status'].apply(
    lambda x: (x == 'Alive').mean() * 100
).sort_values(ascending=True)
axes[2].barh(range(len(marital_surv)), marital_surv.values,
            color=sns.color_palette('Set2', len(marital_surv)), edgecolor='black')
axes[2].set_yticks(range(len(marital_surv)))
axes[2].set_yticklabels([m[:20] for m in marital_surv.index])
axes[2].set_xlabel('Taxa de Sobrevivência (%)')
axes[2].set_title('Sobrevivência por Estado Civil', fontsize=13, fontweight='bold')
for i, v in enumerate(marital_surv.values):
    axes[2].text(v + 0.3, i, f'{v:.1f}%', va='center', fontsize=11)

plt.suptitle('Análise Demográfica e Sobrevivência', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../../reports/figures/seer_demographic_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Análise de Outliers

In [ ]:
# Detecção de outliers usando IQR
print('ANÁLISE DE OUTLIERS (Método IQR)')
print('=' * 60)

outlier_summary = []
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    pct = n_outliers / len(df) * 100
    outlier_summary.append({
        'Feature': col, 'Q1': Q1, 'Q3': Q3, 'IQR': IQR,
        'Lower': lower, 'Upper': upper,
        'N_Outliers': n_outliers, '%': round(pct, 2)
    })
    print(f'{col}: {n_outliers} outliers ({pct:.1f}%) | Limites: [{lower:.1f}, {upper:.1f}]')

pd.DataFrame(outlier_summary)

## 11. Análise Multivariada: Pairplot das Features Principais

In [ ]:
# Pairplot com as variáveis mais correlacionadas
df_plot = df[['Age', 'Tumor Size', 'Regional Node Examined', 
              'Reginal Node Positive', 'Survival Months', 'Status']].copy()
df_plot['Status'] = df_plot['Status'].map({'Alive': 'Viva', 'Dead': 'Óbito'})

g = sns.pairplot(df_plot, hue='Status', 
                 palette={'Viva': '#66c2a5', 'Óbito': '#fc8d62'},
                 diag_kind='kde', plot_kws={'alpha': 0.4, 's': 15})
g.figure.suptitle('Pairplot - Features Numéricas por Status', fontsize=16, fontweight='bold', y=1.02)
plt.savefig('../../reports/figures/seer_pairplot.png', dpi=100, bbox_inches='tight')
plt.show()

## 12. Conclusões da EDA

### Principais Achados:

1. **Dataset desbalanceado**: ~84.7% Alive vs ~15.3% Dead — necessário uso de `class_weight='balanced'` nos modelos

2. **Receptores Hormonais são fortemente preditivos**:
   - Pacientes ER+/PR+ têm melhor sobrevivência
   - Receptores negativos (especialmente duplo-negativo) indicam pior prognóstico

3. **Estadiamento importa significativamente**:
   - T Stage: T4 tem maior taxa de mortalidade
   - N Stage: N3 (mais linfonodos comprometidos) = pior prognóstico
   - 6th Stage: IIIC tem a pior sobrevivência

4. **Grau do tumor**: Tumores pouco diferenciados (Grade III/IV) têm maior mortalidade

5. **Tamanho do tumor**: Tumores maiores correlacionam com pior prognóstico

6. **Sem valores nulos** (exceto Column1 que foi removida): dataset limpo

7. **Outliers presentes** em Tumor Size e Regional Node Examined, mas clinicamente válidos

### Implicações para Modelagem:
- Usar Recall como métrica principal (minimizar falsos negativos em diagnóstico)
- Aplicar balanceamento de classes
- Features de estadiamento e receptores hormonais devem ser priorizadas
- Encoding adequado das variáveis categóricas ordinais (T Stage, N Stage, Grade)